# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

 Method choice: CTR opportunity ranking

My project focuses on identifying content that receives search impressions
but has a low click-through rate relative to comparable content.

The Week-4 baseline ranks content using impressions and CTR shortfall.
For this experiment, I will examine whether a transparent, position-adjusted
CTR opportunity signal can improve prioritization.

The existing trend_direction label measures changes in impressions,
not CTR underperformance. I will not use it as the target for this task.

The dataset contains aggregated 90-day metrics. Consequently, this
experiment is a ranking and signal-analysis exercise unless separate
feature and outcome windows can be established.

I will compare the candidate ranking against the Week-4 baseline on
the same eligible content and use Precision@20 and Precision@50 only
if a defensible relevance label is available.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the repository.
repo = Path.cwd()

while repo != repo.parent and not (repo / "data").exists():
    repo = repo.parent

DATA_PATH = repo / "data" / "raw" / "content_refresh_anonymized.csv"

print("Repository:", repo)
print("Dataset exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Check your working directory."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Repository: c:\Users\Divya\Documents\flyrank-ml
Dataset exists: True
Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



I use a client-grouped holdout split so that content from the same client
does not appear in both training and testing sets.

The target is whether content is marked as declining. The fields
`trend_direction`, `trend_pct`, and `is_declining_label` are excluded
from predictors because they describe or derive from the outcome.

This split evaluates performance on previously unseen clients. However,
the validity of the target and its relationship to the feature window
must be verified before interpreting the results as future prediction.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

required = [
    "client_id",
    "content_id",
    "trend_direction",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

model_df = df.dropna(
    subset=["client_id", "content_id", "trend_direction"]
).copy()

model_df["target"] = (
    model_df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Model rows:", len(model_df))
print("\nTarget distribution:")
print(model_df["target"].value_counts())
print("\nTarget proportions:")
print(model_df["target"].value_counts(normalize=True).round(3))


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["target"],
        groups=model_df["client_id"],
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

assert set(train_df["client_id"]).isdisjoint(
    set(test_df["client_id"])
)

print("\nTraining rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

print("\nClient overlap:",
      len(set(train_df["client_id"]) & set(test_df["client_id"])))

Model rows: 30000

Target distribution:
target
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
target
1    0.542
0    0.458
Name: proportion, dtype: float64

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7

Client overlap: 0


In [5]:

numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_candidates = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
]


numeric_features = [
    c for c in numeric_candidates
    if c in model_df.columns
]

categorical_features = [
    c for c in categorical_candidates
    if c in model_df.columns
]


forbidden = {
    "client_id",
    "content_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "target",
}

features = [
    c for c in numeric_features + categorical_features
    if c not in forbidden
]

print("Features selected:", features)
print("Feature count:", len(features))

assert not (set(features) & forbidden)

Features selected: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used']
Feature count: 24


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    c for c in features
    if c in numeric_candidates
]

categorical_features = [
    c for c in features
    if c in categorical_candidates
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

X_train = train_df[features]
y_train = train_df["target"]

X_test = test_df[features]
y_test = test_df["target"]

model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Random Forest training complete.")

Random Forest training complete.


In [7]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)

predictions = model.predict(X_test)

print("ROC-AUC:",
      round(roc_auc_score(y_test, test_df["model_score"]), 4))

print("Average precision:",
      round(average_precision_score(y_test, test_df["model_score"]), 4))

print("Precision:",
      round(precision_score(y_test, predictions, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_test, predictions, zero_division=0), 4))

ROC-AUC: 0.617
Average precision: 0.6072
Precision: 0.5766
Recall: 0.7028


In [8]:
test_df["predicted_label"] = predictions

false_positives = test_df[
    (test_df["predicted_label"] == 1) &
    (test_df["target"] == 0)
]

false_negatives = test_df[
    (test_df["predicted_label"] == 0) &
    (test_df["target"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

display(
    false_positives[
        ["content_id", "target", "predicted_label", "model_score"]
    ].head(10)
)

display(
    false_negatives[
        ["content_id", "target", "predicted_label", "model_score"]
    ].head(10)
)

False positives: 1625
False negatives: 936


,content_id,target,predicted_label,model_score
13,content_a5a2fbc76336,0,1,0.794148
26,content_72c5c2d73e5a,0,1,0.750278
36,content_bce275871a25,0,1,0.595214
64,content_685de0e3b7cb,0,1,0.610312
78,content_dea0d86223f3,0,1,0.755040
82,content_ec6fce716c78,0,1,0.622765
96,content_600405887c39,0,1,0.654058
126,content_be5e23c0a35e,0,1,0.552505
135,content_670746e86425,0,1,0.585003
181,content_722d8cd002d3,0,1,0.780607


,content_id,target,predicted_label,model_score
0,content_304f48230142,1,0,0.478704
19,content_af865035b328,1,0,0.492031
25,content_033ae3e7aecf,1,0,0.429423
39,content_4595e8704e07,1,0,0.236750
51,content_d8a23b5e10c5,1,0,0.410741
60,content_b9104a222d01,1,0,0.352869
76,content_5607fec5d7db,1,0,0.469072
132,content_cfbb81c693df,1,0,0.491158
148,content_033581b09704,1,0,0.476084
220,content_474bc8a4a3cb,1,0,0.264581


false positives are items predicted as declining that do not carry the decline label, while false negatives are declining items the model misses.


In [2]:
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "search_volume",
    "competition",
    "competition_level",
    "content_type",
    "main_intent",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("All required columns are present.")


All required columns are present.


In [9]:
# Inspecting the columns needed for the CTR opportunity baseline.

inspect_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

available_columns = [
    c for c in inspect_columns
    if c in df.columns
]

print("Available diagnostic columns:")
print(available_columns)

display(df[available_columns].head(10))

print("\nMissing values:")
display(
    df[available_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

print("\nCTR verification:")

if {"clicks_90d", "impressions_90d", "ctr"}.issubset(df.columns):
    valid = df[
        df["impressions_90d"].notna()
        & df["clicks_90d"].notna()
        & df["ctr"].notna()
        & (df["impressions_90d"] > 0)
    ].copy()

    valid["calculated_ctr"] = (
        valid["clicks_90d"] / valid["impressions_90d"] * 100
    )

    valid["ctr_difference"] = (
        valid["ctr"] - valid["calculated_ctr"]
    ).abs()

    print("Rows checked:", len(valid))
    print(
        "Median absolute CTR difference:",
        round(valid["ctr_difference"].median(), 4)
    )

Available diagnostic columns:
['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'trend_direction', 'trend_pct']


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,down,-38.9
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,down,-92.3
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,down,-58.8
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,down,-29.2



Missing values:


,missing_count
content_id,0
client_id,0
impressions_90d,0
clicks_90d,0
ctr,0
avg_position,0
trend_direction,0
trend_pct,3388



CTR verification:
Rows checked: 30000
Median absolute CTR difference: 0.0005


In [3]:
label_candidates = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

for col in label_candidates:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False).head(10))


trend_direction
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct
trend_pct
 NaN      3388
-100.0    1395
 0.0       443
-50.0      281
-66.7      172
 100.0     145
-33.3      137
-75.0       98
-80.0       96
 50.0       88
Name: count, dtype: int64


In [11]:
import pandas as pd
import numpy as np

analysis_df = df[
    df["ctr"].notna()
    & df["avg_position"].notna()
    & df["impressions_90d"].notna()
    & (df["impressions_90d"] > 0)
].copy()


analysis_df["position_bucket"] = pd.cut(
    analysis_df["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    right=True,
    include_lowest=True,
)


position_reference = (
    analysis_df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)

display(position_reference)

,position_bucket,n,median_ctr,mean_ctr,median_impressions
0,1-3,2346,0.00,1.472869,3.0
1,4-10,11842,0.16,0.651045,1184.0
2,11-20,7273,0.10,0.323443,870.0
3,20+,8539,0.00,0.211333,648.0


In [12]:
# Map the reference CTR to each content item.
reference_map = position_reference.set_index(
    "position_bucket"
)["median_ctr"]

analysis_df["reference_ctr"] = (
    analysis_df["position_bucket"].map(reference_map)
)

# Positive values indicate CTR below the position-group median.
analysis_df["ctr_shortfall"] = (
    analysis_df["reference_ctr"] - analysis_df["ctr"]
).clip(lower=0)

# Exclude items without a usable reference.
analysis_df = analysis_df.dropna(
    subset=["reference_ctr", "ctr_shortfall"]
).copy()

# Require at least 500 impressions to focus on visible content.
analysis_df["eligible"] = (
    analysis_df["impressions_90d"] >= 500
)

eligible = analysis_df[
    analysis_df["eligible"]
].copy()

# Transparent rule-based candidate score.
eligible["opportunity_score"] = (
    eligible["impressions_90d"]
    * eligible["ctr_shortfall"]
)

eligible["action"] = np.where(
    eligible["ctr_shortfall"] > 0,
    "REVIEW_CTR",
    "NO_CTR_GAP",
)

eligible = eligible.sort_values(
    "opportunity_score",
    ascending=False,
).reset_index(drop=True)

eligible["rank"] = np.arange(1, len(eligible) + 1)

print("Eligible content:", len(eligible))

display(
    eligible[
        [
            "content_id",
            "impressions_90d",
            "avg_position",
            "ctr",
            "reference_ctr",
            "ctr_shortfall",
            "opportunity_score",
            "action",
            "rank",
        ]
    ].head(10)
)

Eligible content: 16726


,content_id,impressions_90d,avg_position,ctr,reference_ctr,ctr_shortfall,opportunity_score,action,rank
0,content_c8e9d6ab9013,208678,9.7,0.00,0.16,0.16,33388.48,REVIEW_CTR,1
1,content_36ff89c8214e,295097,7.3,0.05,0.16,0.11,32460.67,REVIEW_CTR,2
2,content_c84a0ab98e90,223271,7.8,0.03,0.16,0.13,29025.23,REVIEW_CTR,3
3,content_453722754fea,140079,7.6,0.01,0.16,0.15,21011.85,REVIEW_CTR,4
4,content_c1fe78bc4e37,134055,7.5,0.03,0.16,0.13,17427.15,REVIEW_CTR,5
5,content_39881853ef0c,112434,7.2,0.01,0.16,0.15,16865.10,REVIEW_CTR,6
6,content_0919dd345d80,119217,7.0,0.02,0.16,0.14,16690.38,REVIEW_CTR,7
7,content_b115f7c74779,123469,8.0,0.03,0.16,0.13,16050.97,REVIEW_CTR,8
8,content_91652435f57a,159590,7.8,0.06,0.16,0.10,15959.00,REVIEW_CTR,9
9,content_97a86caf3a3d,147670,6.4,0.07,0.16,0.09,13290.30,REVIEW_CTR,10


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

model_df = df.copy()

for col in ["impressions_90d", "ctr", "avg_position"]:
    model_df[col] = pd.to_numeric(
        model_df[col], errors="coerce"
    )

# Match the Week-4 eligibility rule.
model_df = model_df[
    (model_df["impressions_90d"] >= 500)
    & (model_df["avg_position"] > 0)
    & (model_df["avg_position"] <= 20)
    & model_df["ctr"].notna()
    & model_df["client_id"].notna()
].copy()

# Remove invalid numerical values.
model_df = model_df.replace(
    [np.inf, -np.inf], np.nan
)

model_df = model_df.dropna(
    subset=[
        "impressions_90d",
        "ctr",
        "avg_position",
        "client_id",
    ]
).reset_index(drop=True)

print("Eligible records:", len(model_df))
print("Unique clients:", model_df["client_id"].nunique())

if model_df["client_id"].nunique() < 2:
    raise ValueError(
        "At least two clients are needed for a grouped split."
    )

# Target: observed CTR in the existing 90-day window.
y = model_df["ctr"].astype(float)

# Only use columns that exist in the dataset.
numeric_candidates = [
    "impressions_90d",
    "avg_position",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

categorical_candidates = [
    "content_type",
    "main_intent",
    "competition_level",
]

numeric_features = [
    c for c in numeric_candidates
    if c in model_df.columns
]

categorical_features = [
    c for c in categorical_candidates
    if c in model_df.columns
]

feature_columns = (
    numeric_features + categorical_features
)

if not feature_columns:
    raise ValueError("No usable model features were found.")

# Explicitly exclude target-derived and future/label fields.
forbidden = {
    "ctr",
    "clicks_90d",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

assert not (
    set(feature_columns) & forbidden
), "A target-derived or forbidden field entered the model."

X = model_df[feature_columns].copy()


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=model_df["client_id"],
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(
    model_df.iloc[train_idx]["client_id"]
)

test_clients = set(
    model_df.iloc[test_idx]["client_id"]
)

assert train_clients.isdisjoint(test_clients)

print("Training records:", len(train_idx))
print("Test records:", len(test_idx))
print("Client overlap:", len(train_clients & test_clients))

# Estimate the reference from training data only.
baseline_ctr = float(y_train.median())

baseline_predictions = np.full(
    len(y_test),
    baseline_ctr,
)

print("Week-4 training CTR reference:", baseline_ctr)

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="unknown",
        ),
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

transformers = []

if numeric_features:
    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

results = pd.DataFrame({
    "actual_ctr": y_test.to_numpy(),
    "baseline_prediction": baseline_predictions,
    "model_prediction": model_predictions,
})

comparison = pd.DataFrame([
    {
        "approach": "Week-4 median baseline",
        "MAE": mean_absolute_error(
            results["actual_ctr"],
            results["baseline_prediction"],
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                results["actual_ctr"],
                results["baseline_prediction"],
            )
        ),
        "R2": r2_score(
            results["actual_ctr"],
            results["baseline_prediction"],
        ),
    },
    {
        "approach": "Random Forest",
        "MAE": mean_absolute_error(
            results["actual_ctr"],
            results["model_prediction"],
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                results["actual_ctr"],
                results["model_prediction"],
            )
        ),
        "R2": r2_score(
            results["actual_ctr"],
            results["model_prediction"],
        ),
    },
])

display(comparison.round(5))

Eligible records: 12023
Unique clients: 28
Training records: 11202
Test records: 821
Client overlap: 0
Week-4 training CTR reference: 0.21


,approach,MAE,RMSE,R2
0,Week-4 median baseline,0.31563,0.57750,-0.14115
1,Random Forest,0.32305,0.55224,-0.04350


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The Random Forest model was evaluated against a median-CTR
baseline using the same held-out test records.

The model achieved an MAE of 0.32305, compared with 0.31563
for the baseline. Its RMSE was 0.55224, compared with 0.57750
for the baseline. The model's R² was -0.04350, compared
with -0.14115 for the baseline.

These results are mixed. Random Forest reduced squared
prediction error but did not improve mean absolute error.
Both approaches had negative R² values, so neither
explained the held-out variation particularly well.

Feature importance showed that impressions_90d,
word_count, avg_position, and char_count were the most
important inputs. These associations do not establish
causation.

The largest prediction errors occurred on records with
observed CTR values substantially higher than the model's
predictions. This suggests that the model struggles to
capture some unusually high-CTR cases.

The evaluation held out entire clients, with zero overlap
between training and test clients. This reduces leakage
between clients across the split.

A limitation is that this experiment predicts CTR from
an aggregated 90-day snapshot. It does not establish
whether a content refresh will improve future performance.

In [14]:

results["model_error"] = (
    results["actual_ctr"]
    - results["model_prediction"]
)

results["absolute_error"] = (
    results["model_error"].abs()
)

results["baseline_absolute_error"] = (
    results["actual_ctr"]
    - results["baseline_prediction"]
).abs()

print("Mean absolute error:", results["absolute_error"].mean())

print(
    "Median absolute error:",
    results["absolute_error"].median(),
)

print(
    "90th percentile absolute error:",
    results["absolute_error"].quantile(0.90),
)

print(
    "Model better than baseline on:",
    (
        results["absolute_error"]
        < results["baseline_absolute_error"]
    ).mean().round(4) * 100,
    "% of test records",
)

# Largest errors.
worst_errors = results.assign(
    content_id=model_df.iloc[test_idx]["content_id"].to_numpy(),
    impressions_90d=model_df.iloc[test_idx][
        "impressions_90d"
    ].to_numpy(),
    avg_position=model_df.iloc[test_idx][
        "avg_position"
    ].to_numpy(),
).sort_values(
    "absolute_error",
    ascending=False,
)

display(
    worst_errors[
        [
            "content_id",
            "impressions_90d",
            "avg_position",
            "actual_ctr",
            "baseline_prediction",
            "model_prediction",
            "model_error",
            "absolute_error",
        ]
    ].head(10)
)


fitted_preprocessor = model.named_steps["preprocessor"]

feature_names = fitted_preprocessor.get_feature_names_out()

importances = model.named_steps[
    "regressor"
].feature_importances_

importance_table = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values(
    "importance",
    ascending=False,
)

display(importance_table.head(15))


assert train_clients.isdisjoint(test_clients)

assert not (
    set(feature_columns) & forbidden
)

print("Grouped split check: PASSED")
print("Forbidden-feature check: PASSED")
print("Evaluation records are shared by both approaches.")

Mean absolute error: 0.3230470042551935
Median absolute error: 0.21191551479974272
90th percentile absolute error: 0.6489365818658464
Model better than baseline on: 49.94 % of test records


,content_id,impressions_90d,avg_position,actual_ctr,baseline_prediction,model_prediction,model_error,absolute_error
25,content_ddcbb780b768,793,9.4,5.42,0.21,0.267581,5.152419,5.152419
699,content_a22b7f6c73c5,28192,9.1,4.78,0.21,0.373330,4.406670,4.406670
259,content_97e0945f76a0,2078,4.9,3.56,0.21,0.424238,3.135762,3.135762
74,content_c8ad1f4d0e56,62927,7.2,3.40,0.21,0.382288,3.017712,3.017712
49,content_55add437b24d,3579,3.2,3.10,0.21,0.335005,2.764995,2.764995
72,content_839c5d166ea6,1657,6.1,3.14,0.21,0.430553,2.709447,2.709447
29,content_4d56fd21ca78,4416,4.6,3.03,0.21,0.418465,2.611535,2.611535
175,content_44bad6614cc4,1044,4.4,2.78,0.21,0.347919,2.432081,2.432081
661,content_9217dd688e4b,2309,4.7,2.82,0.21,0.542186,2.277814,2.277814
241,content_a001aa4be7b7,7485,16.2,2.62,0.21,0.345241,2.274759,2.274759


,feature,importance
0,numeric__impressions_90d,0.232246
5,numeric__word_count,0.214753
1,numeric__avg_position,0.207568
6,numeric__char_count,0.180138
2,numeric__search_volume,0.059535
3,numeric__competition,0.040045
4,numeric__cpc,0.020814
11,categorical__main_intent_informational,0.014225
10,categorical__main_intent_commercial,0.010349
13,categorical__main_intent_transactional,0.009641


Grouped split check: PASSED
Forbidden-feature check: PASSED
Evaluation records are shared by both approaches.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Model comparison and error analysis

I compared the Week-4 median-CTR baseline with a Random Forest
regressor using the same held-out records.

The baseline achieved an MAE of [baseline MAE], while the
Random Forest achieved an MAE of [model MAE].

The model [improved/did not improve] on the baseline's MAE.
Its RMSE was [model RMSE], indicating [brief interpretation
of the size of the larger errors].

The largest prediction errors occurred in records with
[describe the observed characteristics of the worst errors].

The most important model features were [list the actual
top features]. These indicate predictive associations,
not causal relationships.

The split held out entire clients, reducing the risk that
content from the same client appears in both training and
test sets.

This is a cross-client evaluation of CTR in an aggregated
90-day snapshot, not a prospective forecast. Future
evaluation should use separate historical feature windows
and subsequent outcome windows when those data are available.